In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

# Load your dataset (assuming it's already loaded as df_mock)
# df_mock = pd.read_csv('your_dataset.csv') # Uncomment if loading from a file

# Display first few rows and info to understand the data structure
print("First few rows of the dataset:\n", df_mock.head())
print("\nDataset Info:\n")
print(df_mock.info())
print("\nSummary statistics:\n", df_mock.describe(include='all').T)

# EDA: Exploratory Data Analysis

# 1. Distribution of Device Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df_mock, x='Device_Type', palette='viridis')
plt.title('Distribution of Device Types')
plt.xlabel('Device Type')
plt.ylabel('Count')
plt.show()

# 2. Distribution of Application Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df_mock, x='Application_Type', palette='plasma')
plt.title('Distribution of Application Types')
plt.xlabel('Application Type')
plt.ylabel('Count')
plt.show()

# 3. Signal Strength (RSSI) by Service Group
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_mock, x='ServiceGroup', y='Signal_Strength_RSSI', palette='coolwarm')
plt.title('Signal Strength (RSSI) Distribution by Service Group')
plt.xlabel('Service Group')
plt.ylabel('Signal Strength (RSSI)')
plt.show()

# 4. Latency Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df_mock['Latency_ms'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Latency')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.show()

# 5. Bandwidth Usage by Application Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_mock, x='Application_Type', y='Bandwidth_Usage_MBps', palette='muted')
plt.title('Bandwidth Usage by Application Type')
plt.xlabel('Application Type')
plt.ylabel('Bandwidth Usage (MBps)')
plt.show()

# 6. Buffer Occupancy by Day of the Week
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_mock, x='Dayoftheweek', y='Buffer_Occupancy', palette='Set2')
plt.title('Buffer Occupancy by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Buffer Occupancy')
plt.show()

# 7. Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df_mock.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# Data Preprocessing

# 1. Handle Missing Values
print("\nChecking for missing values:\n", df_mock.isnull().sum())
# For this mock dataset, there are no missing values, but in real scenarios, you might fill or drop NA values.

# 2. Label Encoding for Categorical Variables
label_encoders = {}
categorical_cols = ['Device_Type', 'Application_Type', 'Time_of_Day', 'ServiceGroup', 'Service', 'Dayoftheweek']

for col in categorical_cols:
    le = LabelEncoder()
    df_mock[col] = le.fit_transform(df_mock[col])
    label_encoders[col] = le  # Store the label encoder for inverse transforming later if needed

print("\nLabel Encoded Categorical Columns:\n", df_mock[categorical_cols].head())

# 3. Standardize Numerical Features
# Selecting numerical features for standardization
numerical_cols = ['Signal_Strength_RSSI', 'Latency_ms', 'Jitter_ms', 'Packet_Loss_Rate', 'Bandwidth_Usage_MBps', 'Buffer_Occupancy']
scaler = StandardScaler()
df_mock[numerical_cols] = scaler.fit_transform(df_mock[numerical_cols])

print("\nStandardized Numerical Columns:\n", df_mock[numerical_cols].head())

# 4. Drop Unnecessary Columns
# Drop columns that are not needed for modeling (like Hashid, Timestamp)
df_mock = df_mock.drop(columns=['Hashid', 'Timestamp'])

print("\nFinal Preprocessed Dataset:\n", df_mock.head())


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
from datetime import datetime

# Assume df_mock is your dataset

# Descriptive Analysis
print("Descriptive Analysis:\n")

# 1. Summary statistics for numerical columns
print("\nSummary Statistics for Numerical Columns:\n", df_mock.describe().T)

# 2. Frequency counts for categorical columns
categorical_cols = ['Device_Type', 'Application_Type', 'Time_of_Day', 'ServiceGroup', 'Service', 'Dayoftheweek']
for col in categorical_cols:
    print(f"\nFrequency count for {col}:\n", df_mock[col].value_counts())

# 3. Summary statistics by Application_Type for key metrics
print("\nSummary of Bandwidth Usage by Application Type:\n", df_mock.groupby('Application_Type')['Bandwidth_Usage_MBps'].describe())
print("\nSummary of Latency by Application Type:\n", df_mock.groupby('Application_Type')['Latency_ms'].describe())

# Statistical Analysis
print("\nStatistical Analysis:\n")

# 1. Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df_mock.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# 2. Variance and standard deviation of numerical features
variance = df_mock.var(numeric_only=True)
std_dev = df_mock.std(numeric_only=True)
print("Variance of Numerical Features:\n", variance)
print("\nStandard Deviation of Numerical Features:\n", std_dev)

# Inferential Analysis
print("\nInferential Analysis:\n")

# 1. T-Test for Latency between Video Conference and Streaming Applications
latency_video_conference = df_mock[df_mock['Application_Type'] == 'Video Conference']['Latency_ms']
latency_streaming = df_mock[df_mock['Application_Type'].isin(['YouTube Streaming', 'Netflix Streaming'])]['Latency_ms']
t_stat, p_val = ttest_ind(latency_video_conference, latency_streaming)

print("T-Test Results for Latency (Video Conference vs. Streaming):")
print("T-statistic:", t_stat)
print("P-value:", p_val)
if p_val < 0.05:
    print("Result: Significant difference in latency between video conferencing and streaming applications.")
else:
    print("Result: No significant difference in latency between video conferencing and streaming applications.")

# 2. Chi-Square Test: Relationship Between Traffic Spike and Application Type
contingency_table = pd.crosstab(df_mock['Traffic_Spike'], df_mock['Application_Type'])
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("\nChi-Square Test Results (Traffic Spike vs Application Type):")
print("Chi-square statistic:", chi2)
print("P-value:", p)
if p < 0.05:
    print("Result: Significant relationship between traffic spike and application type.")
else:
    print("Result: No significant relationship between traffic spike and application type.")

# 3. ANOVA Test for Bandwidth Usage Across Application Types
bandwidth_conference = df_mock[df_mock['Application_Type'] == 'Video Conference']['Bandwidth_Usage_MBps']
bandwidth_youtube = df_mock[df_mock['Application_Type'] == 'YouTube Streaming']['Bandwidth_Usage_MBps']
bandwidth_netflix = df_mock[df_mock['Application_Type'] == 'Netflix Streaming']['Bandwidth_Usage_MBps']
f_stat, p_val_anova = f_oneway(bandwidth_conference, bandwidth_youtube, bandwidth_netflix)

print("\nANOVA Results for Bandwidth Usage Across Application Types:")
print("F-statistic:", f_stat)
print("P-value:", p_val_anova)
if p_val_anova < 0.05:
    print("Result: Significant differences in bandwidth usage across application types.")
else:
    print("Result: No significant differences in bandwidth usage across application types.")

# Preprocessing (as in the previous code)

# 1. Label Encoding for Categorical Variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_mock[col] = le.fit_transform(df_mock[col])
    label_encoders[col] = le

# 2. Standardize Numerical Features
numerical_cols = ['Signal_Strength_RSSI', 'Latency_ms', 'Jitter_ms', 'Packet_Loss_Rate', 'Bandwidth_Usage_MBps', 'Buffer_Occupancy']
scaler = StandardScaler()
df_mock[numerical_cols] = scaler.fit_transform(df_mock[numerical_cols])

# 3. Drop Unnecessary Columns
df_mock = df_mock.drop(columns=['Hashid', 'Timestamp'])

print("\nPreprocessed Dataset:\n", df_mock.head())
